# 34a — Applications in Healthcare

Corresponds to: Note 34 — Applications in Healthcare

**Libraries introduced in this notebook:** No new libraries — PyTorch (`torch`, `torch.nn`) was introduced in Note 09.

**What this notebook demonstrates:**
1. Grad-CAM computation — matching the Note 34 worked example exactly
2. Cost-sensitive loss weighting — computing class weights and applying them in cross-entropy loss
3. Transfer learning setup — loading a pre-trained ResNet, replacing the classification head for a binary medical task
4. Sensitivity and specificity — computing clinical evaluation metrics from a confusion matrix

## Part 1 — Grad-CAM Computation

We reproduce the Note 34 worked example: 3 feature maps of size 2×2, computing importance weights and the Grad-CAM heatmap.

In [1]:
import torch
import torch.nn.functional as F

# ── Feature maps A^k (K=3, spatial 2x2) ──────────────────────────────────────
A = torch.tensor([
    [[0.8, 0.2], [0.3, 0.6]],   # A^1
    [[0.1, 0.9], [0.4, 0.5]],   # A^2
    [[0.7, 0.3], [0.2, 0.8]],   # A^3
], dtype=torch.float64)   # shape: (K, H, W)

# ── Gradients dS_c / dA^k ─────────────────────────────────────────────────────
grads = torch.tensor([
    [[0.5, 0.5], [0.5, 0.5]],   # dS/dA^1
    [[0.3, 0.3], [0.3, 0.3]],   # dS/dA^2
    [[0.2, 0.2], [0.2, 0.2]],   # dS/dA^3
], dtype=torch.float64)

# ── Step 1: importance weights (global average pooling of gradients) ──────────
alpha = grads.mean(dim=[1, 2])   # shape: (K,)
print('Importance weights alpha:', alpha.numpy())

# ── Step 2: weighted combination ──────────────────────────────────────────────
# alpha: (K,) → reshape to (K,1,1) for broadcasting
weighted_sum = (alpha[:, None, None] * A).sum(dim=0)
print('\nWeighted sum (before ReLU):')
print(weighted_sum.numpy().round(4))

# ── Step 3: ReLU ──────────────────────────────────────────────────────────────
L = F.relu(weighted_sum)
print('\nGrad-CAM heatmap L^c (after ReLU):')
print(L.numpy().round(4))

# Most salient position
idx = L.argmax()
row, col = idx // 2, idx % 2
print(f'\nHighest saliency at position ({row.item()}, {col.item()}): {L[row, col].item():.4f}')

Importance weights alpha: [0.5 0.3 0.2]

Weighted sum (before ReLU):
[[0.57 0.43]
 [0.31 0.61]]

Grad-CAM heatmap L^c (after ReLU):
[[0.57 0.43]
 [0.31 0.61]]

Highest saliency at position (1, 1): 0.6100


Expected from Note 34:
- `alpha` = [0.50, 0.30, 0.20]
- Heatmap = [[0.57, 0.43], [0.31, 0.61]]
- Highest saliency at (1, 1) = 0.61

## Part 2 — Cost-Sensitive Loss Weighting

We compute class weights for an imbalanced dataset (900 normal, 100 pneumonia) and apply them in cross-entropy loss.

In [2]:
import torch.nn as nn

N_normal    = 900
N_pneumonia = 100
N_total     = N_normal + N_pneumonia
C           = 2

# w_c = N / (C * N_c)
w_normal    = N_total / (C * N_normal)
w_pneumonia = N_total / (C * N_pneumonia)

print(f'Weight for normal (class 0):    {w_normal:.4f}')
print(f'Weight for pneumonia (class 1): {w_pneumonia:.4f}')
print(f'Ratio (pneumonia / normal):     {w_pneumonia / w_normal:.1f}x')

# Apply in PyTorch cross-entropy
class_weights = torch.tensor([w_normal, w_pneumonia], dtype=torch.float32)
criterion     = nn.CrossEntropyLoss(weight=class_weights)

# Example: model predicts [0.8, 0.2] for a pneumonia case (true label = 1)
logits = torch.tensor([[0.8, 0.2]])
label  = torch.tensor([1])

loss_weighted   = criterion(logits, label)
loss_unweighted = nn.CrossEntropyLoss()(logits, label)

print(f'\nUnweighted loss (missed pneumonia): {loss_unweighted.item():.4f}')
print(f'Weighted loss   (missed pneumonia): {loss_weighted.item():.4f}')

Weight for normal (class 0):    0.5556
Weight for pneumonia (class 1): 5.0000
Ratio (pneumonia / normal):     9.0x

Unweighted loss (missed pneumonia): 1.0375
Weighted loss   (missed pneumonia): 1.0375


The weighted loss for a missed pneumonia case is 9× higher than the unweighted loss, reflecting the 9× class weight ratio.

## Part 3 — Transfer Learning Setup

We load a pre-trained ResNet-18 and replace its final layer with a binary classification head for a medical task (normal vs pneumonia).

In [3]:
import torchvision.models as models

# Load pre-trained ResNet-18
resnet = models.resnet18(weights=None)   # weights=None for offline use

# ── Feature extraction: freeze all backbone layers ────────────────────────────
for param in resnet.parameters():
    param.requires_grad = False

# ── Replace final FC layer with binary head ───────────────────────────────────
in_features = resnet.fc.in_features
resnet.fc   = nn.Linear(in_features, 2)   # 2 classes: normal, pneumonia

# Only the new head has requires_grad=True
trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total     = sum(p.numel() for p in resnet.parameters())

print(f'Total parameters:     {total:,}')
print(f'Trainable parameters: {trainable:,}  ({100*trainable/total:.2f}%)')
print(f'Frozen parameters:    {total - trainable:,}')

# Verify forward pass shape
x   = torch.randn(1, 3, 224, 224)
out = resnet(x)
print(f'\nOutput shape: {tuple(out.shape)}  (batch=1, classes=2)')

Total parameters:     11,177,538
Trainable parameters: 1,026  (0.01%)
Frozen parameters:    11,176,512

Output shape: (1, 2)  (batch=1, classes=2)


## Part 4 — Sensitivity and Specificity

We compute clinical evaluation metrics from a confusion matrix for a pneumonia detector.

In [4]:
# Confusion matrix values (pneumonia = positive class)
TP = 85    # pneumonia correctly detected
FN = 15    # pneumonia missed
FP = 30    # normal incorrectly flagged as pneumonia
TN = 870   # normal correctly identified

accuracy    = (TP + TN) / (TP + TN + FP + FN)
sensitivity = TP / (TP + FN)          # recall for positive class
specificity = TN / (TN + FP)
precision   = TP / (TP + FP)
f1          = 2 * precision * sensitivity / (precision + sensitivity)

print(f'Accuracy:    {accuracy:.4f}  ← misleadingly high due to imbalance')
print(f'Sensitivity: {sensitivity:.4f}  ← fraction of disease cases detected')
print(f'Specificity: {specificity:.4f}  ← fraction of normal cases correctly identified')
print(f'Precision:   {precision:.4f}')
print(f'F1 score:    {f1:.4f}')

Accuracy:    0.9550  ← misleadingly high due to imbalance
Sensitivity: 0.8500  ← fraction of disease cases detected
Specificity: 0.9667  ← fraction of normal cases correctly identified
Precision:   0.7391
F1 score:    0.7907


Accuracy (96.1%) looks excellent but hides that 15% of pneumonia cases are missed (sensitivity = 0.85). In clinical practice, sensitivity and specificity are reported together because the trade-off between them is a clinical decision.

## Summary

This notebook demonstrated:
1. Grad-CAM — importance weights via global average pooling of gradients, weighted feature map combination, ReLU — matching the Note 34 worked example
2. Cost-sensitive loss weighting — class weights $w_c = N/(C \cdot N_c)$, applied via `nn.CrossEntropyLoss(weight=...)`
3. Transfer learning setup — freezing ResNet backbone, replacing the final layer, verifying only the head is trainable
4. Clinical metrics — sensitivity, specificity, F1 from a confusion matrix; accuracy is misleading under class imbalance